# Assignment 8: Build the Intent Dataset for Your Own Chatbot
### AITCD001 Chatbots Development · Learning Outcome 3 · United African Institute of Technology

**Due Monday 02nd November 2026, 23:59**

Cells marked **YOUR DOMAIN** are replaced with your own chatbot's data. Cells marked **PROVIDED**
do the checking, the split, the baseline and the entity count: read them, do not change them.

The worked example is a **fictional** bus ticket desk (the Week 7 example) with only 6 messages
per label, so the quality report shows what is still missing. It may not be submitted.

**Steps:** Save a copy of this file in Drive. Replace Steps 1, 2, 3, 6 and 7 with your own, and run
everything from the top (Runtime, then Run all). Repeat Steps 3 and 4 until the report is clean.

---

---

## Setup

**PROVIDED.** Run once. The same spaCy model as the session notebook.

In [ ]:
# AITCD001, Week 8 take home: the intent dataset for my own chatbot
import csv
import random
import re
from collections import Counter

import pandas as pd
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:                  # not installed yet: install it once
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("ready: spaCy", spacy.__version__, "| model", nlp.meta["version"])

ready: spaCy 3.8.16 | model 3.8.0


---

## Step 1: The chatbot and its labels

**YOUR DOMAIN.** The chatbot of your Portfolio Project 1. All of its intents, plus `out_of_scope`. Do not add `greeting` or `name` unless your users really ask for something with them.

In [ ]:
# STEP 1. YOUR DOMAIN: the chatbot of your Portfolio Project 1, and its labels.
BOT_NAME = "UAIT AI"
INTENTS = ["cancel", "callback", "fees", "programs", "apply", "location", "homework", "schedule", "exams", "name", "greeting", "out_of_scope"]

TARGET = 30          # messages per label, at least
print(BOT_NAME, "|", len(INTENTS), "labels:", INTENTS)


City Bus Desk | 6 labels: ['hours', 'fares', 'luggage', 'book', 'cancel', 'out_of_scope']


---

## Step 2: The labelling guideline

**YOUR DOMAIN.** Write the guideline BEFORE labelling. Every time two labels are possible for a message, add or sharpen a rule.

In [ ]:
# STEP 2. YOUR DOMAIN: your labelling guideline.
GUIDELINE = [
    "One message, one label; split a message if it contains two separate requests.",
    "Label the user's main goal, not just a keyword. Questions about tuition, prices or payment amounts are fees.",
    "Questions about available courses or what a course teaches are programs; asking to enroll or register is apply.",
    "Requests for a phone call or appointment with staff are callback; withdrawing that request is cancel.",
    "Questions about campus address or directions are location.",
    "Homework, assignment, portfolio and submission deadlines are homework; class times and timetables are schedule; tests, grades and exam results are exams.",
    "Use name only when the user introduces their own name; use greeting for a greeting without a more specific request.",
    "Use out_of_scope when the request is unrelated to UAIT admissions, programs, fees, campus, academic work or contacting the institute.",
    "Preserve the message as written, including spelling, SMS forms and Kinyarwanda/mixed-language phrasing.",
    "All examples in this version are constructed practice examples, not verified messages collected from real people.",
]

for i, rule in enumerate(GUIDELINE, start=1):
    print(i, rule)


1 One message, one label. Two requests in one message: split it first.
2 Label the goal, not the words: 'how much is the bag fee' is luggage.
3 Booking anything is book, even with a town or a day in it.
4 A fare question about a town is fares, not book.
5 Complaints, lost property, jobs, a person: out_of_scope.
6 Keep the traveller's words: typos, SMS, Kinyarwanda, as typed.


---

## Step 3: The dataset

**YOUR DOMAIN.** At least 30 messages per label. Sources, in order of value: your chatbot's UNMATCHED log and Step 10 local phrasing from Portfolio Project 1; at least 20 messages from real people you ask (classmates, family, friends, for example on WhatsApp: *how would you ask a bus desk about luggage?*); then your own. Keep every message exactly as it was typed.

In [ ]:
# STEP 3. YOUR DOMAIN: 360 constructed practice messages (30 per intent).
# Source labels are honest: rules = examples based on existing rule patterns; local = constructed Kinyarwanda/SMS-style examples; written = other constructed examples.
# No invented example is labelled 'people' or 'unmatched'.
DATA = [
    ("cancel my callback request", "cancel", "rules"),
    ("please cancel the appointment", "cancel", "rules"),
    ("cancle my request", "cancel", "rules"),
    ("call it off please", "cancel", "rules"),
    ("never mind about the callback", "cancel", "rules"),
    ("ndashaka guhagarika ubusabe bwanjye", "cancel", "local"),
    ("reka guhagarika gahunda yari yasabwe", "cancel", "local"),
    ("ntimuzampamagare nkiri kubihagaritse", "cancel", "local"),
    ("ndahinduye ibitekerezo, mubihagarike", "cancel", "local"),
    ("guhagarika appointment yanjye", "cancel", "local"),
    ("I no longer need someone to call me", "cancel", "written"),
    ("remove my callback from the list", "cancel", "written"),
    ("I changed my mind about speaking to admissions", "cancel", "written"),
    ("stop the call request I made earlier", "cancel", "written"),
    ("please withdraw my request for a phone call", "cancel", "written"),
    ("I don't want the appointment anymore", "cancel", "written"),
    ("forget the call I asked for", "cancel", "written"),
    ("can you delete my callback request", "cancel", "written"),
    ("I will contact the school myself instead", "cancel", "written"),
    ("please void the call arrangement", "cancel", "written"),
    ("cancel the call scheduled for me", "cancel", "written"),
    ("I won't be available for that callback so cancel it", "cancel", "written"),
    ("please take my name off the callback list", "cancel", "written"),
    ("I want to withdraw the appointment request", "cancel", "written"),
    ("no need to call me anymore", "cancel", "written"),
    ("kindly cancel the phone consultation", "cancel", "written"),
    ("I don't want to continue with the request", "cancel", "written"),
    ("undo my callback booking", "cancel", "written"),
    ("please stop the appointment process", "cancel", "written"),
    ("I have decided not to receive a call", "cancel", "written"),
    ("can I get a callback", "callback", "rules"),
    ("please call me", "callback", "rules"),
    ("I need an appointment with admissions", "callback", "rules"),
    ("contact me about enrollment", "callback", "rules"),
    ("request a call back from the office", "callback", "rules"),
    ("ndashaka ko mwampamagara", "callback", "local"),
    ("mwampamagara mukansobanurira?", "callback", "local"),
    ("ndashaka appointment n'abashinzwe admissions", "callback", "local"),
    ("mumpamagare kuri application yanjye", "callback", "local"),
    ("mwambwira igihe mwampamagarira?", "callback", "local"),
    ("someone from UAIT should phone me", "callback", "written"),
    ("I would like to speak to an admissions officer", "callback", "written"),
    ("please arrange a phone consultation", "callback", "written"),
    ("can the school call me this afternoon", "callback", "written"),
    ("I want an advisor to reach me by phone", "callback", "written"),
    ("how do I request a phone call from UAIT", "callback", "written"),
    ("please have your team contact me", "callback", "written"),
    ("I need a call to discuss the courses", "callback", "written"),
    ("could you schedule a call with admissions", "callback", "written"),
    ("I prefer to explain my question over the phone", "callback", "written"),
    ("please ring me when someone is free", "callback", "written"),
    ("I want the office to call me back tomorrow", "callback", "written"),
    ("can I book a time to talk by phone", "callback", "written"),
    ("I need someone to contact me about fees", "callback", "written"),
    ("please set up a callback for me", "callback", "written"),
    ("would an admissions representative call me", "callback", "written"),
    ("I want to talk with a staff member", "callback", "written"),
    ("please phone me regarding registration", "callback", "written"),
    ("can the school reach me on my number", "callback", "written"),
    ("I need a callback about my application", "callback", "written"),
    ("what are the fees", "fees", "rules"),
    ("how much does tuition cost", "fees", "rules"),
    ("tell me the prices for the programs", "fees", "rules"),
    ("hw much is the application fee", "fees", "rules"),
    ("minerval ni angahe", "fees", "local"),
    ("amafaranga y'ishuri ni angahe", "fees", "local"),
    ("kwiga AI technology bisaba angahe", "fees", "local"),
    ("amafaranga yo kwiyandikisha ni angahe", "fees", "local"),
    ("minerval ya cyber security imeze ite", "fees", "local"),
    ("mushobora kumbwira amafaranga y'amasomo?", "fees", "local"),
    ("how much should I pay to join UAIT", "fees", "written"),
    ("what is the total cost for one semester", "fees", "written"),
    ("is tuition paid monthly or per term", "fees", "written"),
    ("do you have a payment plan for students", "fees", "written"),
    ("how much is the registration charge", "fees", "written"),
    ("are there extra costs apart from tuition", "fees", "written"),
    ("what is the fee for the AI course", "fees", "written"),
    ("please send me the current fee structure", "fees", "written"),
    ("does the school offer scholarships or discounts", "fees", "written"),
    ("what amount do I need before classes start", "fees", "written"),
    ("how much is cyber security training", "fees", "written"),
    ("I need details about tuition and other charges", "fees", "written"),
    ("is the application fee separate from tuition", "fees", "written"),
    ("can I pay school fees in installments", "fees", "written"),
    ("what is the price of drone technology", "fees", "written"),
    ("do fees differ between programs", "fees", "written"),
    ("how much money is required for enrollment", "fees", "written"),
    ("where can I see the official fee schedule", "fees", "written"),
    ("is there a deposit before admission", "fees", "written"),
    ("what are the charges for jewelry design", "fees", "written"),
    ("what programs do you offer", "programs", "rules"),
    ("tell me about cyber security", "programs", "rules"),
    ("wat progs u hav", "programs", "local"),
    ("ni ayahe masomo mutanga", "programs", "local"),
    ("amasomo ya AI technology arimo iki", "programs", "local"),
    ("do you teach drone technology", "programs", "written"),
    ("which courses can I study at UAIT", "programs", "written"),
    ("I want to know about jewelry design", "programs", "written"),
    ("is piano performance available", "programs", "written"),
    ("what can I study at your institute", "programs", "written"),
    ("please list all available programs", "programs", "written"),
    ("do you have an artificial intelligence course", "programs", "written"),
    ("tell me what the cyber security program covers", "programs", "written"),
    ("are there practical courses in technology", "programs", "written"),
    ("what subjects are included in drone technology", "programs", "written"),
    ("I am interested in the AI program", "programs", "written"),
    ("does UAIT teach creative arts", "programs", "written"),
    ("which program is best for someone interested in robotics", "programs", "written"),
    ("can you explain the jewelry design course", "programs", "written"),
    ("do you offer a piano course", "programs", "written"),
    ("show me the courses available this year", "programs", "written"),
    ("I want details about your technology programs", "programs", "written"),
    ("what are the study options at the institute", "programs", "written"),
    ("is cybersecurity one of your courses", "programs", "written"),
    ("do you have a course about artificial intelligence", "programs", "written"),
    ("what programs are accepting students", "programs", "written"),
    ("I would like a description of each course", "programs", "written"),
    ("are there short courses at UAIT", "programs", "written"),
    ("can you tell me about the drone course", "programs", "written"),
    ("which programs does the school provide", "programs", "written"),
    ("how do I apply", "apply", "rules"),
    ("I want to join UAIT", "apply", "rules"),
    ("where is the application form", "apply", "rules"),
    ("kwiyandikisha bikorwa bite", "apply", "local"),
    ("ndashaka gusaba kwiga muri UAIT", "apply", "local"),
    ("what documents are needed for admission", "apply", "written"),
    ("how can I register as a new student", "apply", "written"),
    ("when does enrollment open", "apply", "written"),
    ("I want to submit an application", "apply", "written"),
    ("can I apply online", "apply", "written"),
    ("please guide me through the admission process", "apply", "written"),
    ("where do I send my certificates", "apply", "written"),
    ("what are the requirements to join", "apply", "written"),
    ("is registration still open", "apply", "written"),
    ("I would like to become a student", "apply", "written"),
    ("how do I enroll for AI technology", "apply", "written"),
    ("can you share the admissions link", "apply", "written"),
    ("what is the first step to get admitted", "apply", "written"),
    ("I need help completing my application", "apply", "written"),
    ("when is the next intake", "apply", "written"),
    ("do I need previous qualifications to apply", "apply", "written"),
    ("how can international applicants register", "apply", "written"),
    ("I want to start the enrollment process", "apply", "written"),
    ("which papers should I prepare for admission", "apply", "written"),
    ("where can I get an application form", "apply", "written"),
    ("gusaba kwiga bisaba iki", "apply", "local"),
    ("kwinjira muri institute bikorwa gute", "apply", "local"),
    ("nkeneye kwiyandikisha muri iyi gahunda", "apply", "local"),
    ("ni ryari kwakira abanyeshuri bitangira", "apply", "local"),
    ("mwampa amabwiriza yo kwiyandikisha", "apply", "local"),
    ("where is UAIT", "location", "rules"),
    ("what is your address", "location", "rules"),
    ("wer is the campus", "location", "rules"),
    ("campus iri he", "location", "local"),
    ("ishuri riri he i Kigali", "location", "local"),
    ("how do I get to the institute", "location", "written"),
    ("which area of Kigali are you located in", "location", "written"),
    ("is the campus in Kimihurura", "location", "written"),
    ("please share directions to UAIT", "location", "written"),
    ("where can I find your offices", "location", "written"),
    ("send me the location pin", "location", "written"),
    ("what district is the school in", "location", "written"),
    ("is UAIT near the city center", "location", "written"),
    ("how can I reach the campus by taxi", "location", "written"),
    ("I need the physical address of the institute", "location", "written"),
    ("where should I come for admission", "location", "written"),
    ("can you tell me the campus location", "location", "written"),
    ("which neighborhood is UAIT in", "location", "written"),
    ("is your office in Gasabo", "location", "written"),
    ("give me directions from Kigali Convention Centre", "location", "written"),
    ("muri Kimihurura he?", "location", "local"),
    ("mwashyira hano location ya campus", "location", "local"),
    ("UAIT iherereye mu wuhe murenge", "location", "local"),
    ("ni gute nageza kuri campus", "location", "local"),
    ("aho mwigishiriza ni hehe", "location", "local"),
    ("I am trying to visit the school today", "location", "written"),
    ("what landmark is close to your campus", "location", "written"),
    ("can I visit your admissions office", "location", "written"),
    ("please tell me where the institute is based", "location", "written"),
    ("is the campus address in Kigali", "location", "written"),
    ("when is the homework due", "homework", "rules"),
    ("what is the assignment deadline", "homework", "rules"),
    ("when should I submit my portfolio", "homework", "rules"),
    ("akazi ko murugo kazatangwa ryari", "homework", "local"),
    ("assignment igomba gutangwa ryari", "homework", "local"),
    ("is the AITCD001 task due this Thursday", "homework", "written"),
    ("where do I upload my homework", "homework", "written"),
    ("I need the deadline for this week's work", "homework", "written"),
    ("has the new assignment been released", "homework", "written"),
    ("what happens if I submit late", "homework", "written"),
    ("can you remind me of the portfolio deadline", "homework", "written"),
    ("which platform should I use to submit", "homework", "written"),
    ("is the homework due before midnight", "homework", "written"),
    ("I cannot see the assignment instructions", "homework", "written"),
    ("when is the next task expected", "homework", "written"),
    ("do we have homework to complete this week", "homework", "written"),
    ("please explain how to submit the project", "homework", "written"),
    ("what is the last date for uploading the assignment", "homework", "written"),
    ("is there a practical task for AITCD001", "homework", "written"),
    ("I missed the homework deadline what should I do", "homework", "written"),
    ("mwadufasha kumenya deadline ya assignment", "homework", "local"),
    ("portfolio tuyohereza ryari", "homework", "local"),
    ("homework ya AITCD001 izarangira ryari", "homework", "local"),
    ("ese hari umukoro mushya", "homework", "local"),
    ("assignment tuyishyira he", "homework", "local"),
    ("can I submit my work through email", "homework", "written"),
    ("where are the homework guidelines", "homework", "written"),
    ("what time exactly is the deadline", "homework", "written"),
    ("is the project submission open now", "homework", "written"),
    ("please tell me the due date for the portfolio", "homework", "written"),
    ("what time is class", "schedule", "rules"),
    ("please send the timetable", "schedule", "rules"),
    ("wat tym are lectures", "schedule", "local"),
    ("amasomo atangira saa ngahe", "schedule", "local"),
    ("schedule yacu iri he", "schedule", "local"),
    ("which days do we attend lectures", "schedule", "written"),
    ("are there classes on Saturday", "schedule", "written"),
    ("what is the timetable for AI students", "schedule", "written"),
    ("when does the afternoon session begin", "schedule", "written"),
    ("how many hours do we study each day", "schedule", "written"),
    ("I need to know my class schedule", "schedule", "written"),
    ("what time should I arrive at campus", "schedule", "written"),
    ("are lessons held in the evening", "schedule", "written"),
    ("when is the next lecture", "schedule", "written"),
    ("can you share this week's class calendar", "schedule", "written"),
    ("do different programs have different schedules", "schedule", "written"),
    ("what time does the morning class start", "schedule", "written"),
    ("is there a break between lectures", "schedule", "written"),
    ("where can I check the weekly timetable", "schedule", "written"),
    ("are classes online or on campus", "schedule", "written"),
    ("ni ryari dutangira kwiga", "schedule", "local"),
    ("amasomo ya cyber security aba ryari", "schedule", "local"),
    ("mwaduha timetable y'iki cyumweru", "schedule", "local"),
    ("ku wa gatanu hari lecture", "schedule", "local"),
    ("ni saa ngahe tugomba kugera kuri campus", "schedule", "local"),
    ("how often do students attend classes", "schedule", "written"),
    ("does the institute have evening lessons", "schedule", "written"),
    ("what is the schedule for this semester", "schedule", "written"),
    ("are lectures happening tomorrow", "schedule", "written"),
    ("please confirm the class starting time", "schedule", "written"),
    ("when are the exams", "exams", "rules"),
    ("how can I check my results", "exams", "rules"),
    ("what are my marks", "exams", "rules"),
    ("ibizamini bizaba ryari", "exams", "local"),
    ("amanota yanjye nayabona nte", "exams", "local"),
    ("has the exam timetable been published", "exams", "written"),
    ("where do students collect their grades", "exams", "written"),
    ("when will results be released", "exams", "written"),
    ("do we have a test this week", "exams", "written"),
    ("I want to know my examination date", "exams", "written"),
    ("how are final marks calculated", "exams", "written"),
    ("can I retake an exam if I fail", "exams", "written"),
    ("when is the next assessment", "exams", "written"),
    ("please explain the exam rules", "exams", "written"),
    ("are results available online", "exams", "written"),
    ("I cannot find my exam score", "exams", "written"),
    ("what date is the final examination", "exams", "written"),
    ("will the lecturer announce the grades", "exams", "written"),
    ("is there a resit examination", "exams", "written"),
    ("how do I appeal an exam result", "exams", "written"),
    ("amanota arasohoka ryari", "exams", "local"),
    ("tuzakora test ryari", "exams", "local"),
    ("nabuze result y'ikizamini", "exams", "local"),
    ("ese exam timetable yarasohotse", "exams", "local"),
    ("ibizamini bya semester ni ryari", "exams", "local"),
    ("where can I see my assessment results", "exams", "written"),
    ("do we have practical exams", "exams", "written"),
    ("when should I expect my transcript", "exams", "written"),
    ("is the exam postponed", "exams", "written"),
    ("what is the procedure after exam results", "exams", "written"),
    ("my name is Aline", "name", "rules"),
    ("my name is Eric", "name", "rules"),
    ("my name is Jean", "name", "rules"),
    ("nitwa Aline", "name", "local"),
    ("nitwa Eric kandi ndi umunyeshuri", "name", "local"),
    ("you can call me Patrick", "name", "written"),
    ("I am called Diane", "name", "written"),
    ("I would like to introduce myself as Kevin", "name", "written"),
    ("my full name is Grace Uwase", "name", "written"),
    ("the name I use is Samuel", "name", "written"),
    ("I'm Alice", "name", "written"),
    ("I am Brian from Kigali", "name", "written"),
    ("please call me Emmanuel", "name", "written"),
    ("my first name is Clarisse", "name", "written"),
    ("I go by the name David", "name", "written"),
    ("for this chat my name is Esther", "name", "written"),
    ("my name is Beata and I need information", "name", "written"),
    ("I am known as Thierry", "name", "written"),
    ("people call me Sandrine", "name", "written"),
    ("this is Claude speaking", "name", "written"),
    ("izina ryanjye ni Marie", "name", "local"),
    ("njye nitwa Fabrice", "name", "local"),
    ("ndi Diane", "name", "local"),
    ("banyita Kevin", "name", "local"),
    ("nitwa Sandrine wo muri Kigali", "name", "local"),
    ("hello my name is Alice", "name", "written"),
    ("my name is Jean Pierre", "name", "written"),
    ("I should introduce myself as Chantal", "name", "written"),
    ("you may address me as Robert", "name", "written"),
    ("I am called Olive", "name", "written"),
    ("hello", "greeting", "rules"),
    ("hi there", "greeting", "rules"),
    ("hey", "greeting", "rules"),
    ("muraho", "greeting", "local"),
    ("mwaramutse", "greeting", "local"),
    ("mwiriwe neza", "greeting", "local"),
    ("amakuru yawe", "greeting", "local"),
    ("bite se", "greeting", "local"),
    ("hello UAIT", "greeting", "written"),
    ("good morning", "greeting", "written"),
    ("good afternoon", "greeting", "written"),
    ("good evening", "greeting", "written"),
    ("hi I need some information", "greeting", "written"),
    ("hey there assistant", "greeting", "written"),
    ("hello how are you", "greeting", "written"),
    ("greetings", "greeting", "written"),
    ("is anyone there", "greeting", "written"),
    ("hey can you help me", "greeting", "written"),
    ("good day", "greeting", "written"),
    ("hello team", "greeting", "written"),
    ("muraho neza", "greeting", "local"),
    ("mwaramutse neza", "greeting", "local"),
    ("mwiriwe neza UAIT", "greeting", "local"),
    ("amakuru", "greeting", "local"),
    ("bite", "greeting", "local"),
    ("hi chatbot", "greeting", "written"),
    ("hello there", "greeting", "written"),
    ("hey UAIT assistant", "greeting", "written"),
    ("good morning everyone", "greeting", "written"),
    ("how is your day", "greeting", "written"),
    ("what is the weather today", "out_of_scope", "written"),
    ("who won the football match", "out_of_scope", "written"),
    ("can you help me find a job", "out_of_scope", "written"),
    ("I lost my phone yesterday", "out_of_scope", "written"),
    ("how do I cook rice", "out_of_scope", "written"),
    ("tell me a joke", "out_of_scope", "written"),
    ("what is the capital of Japan", "out_of_scope", "written"),
    ("can you book a hotel for me", "out_of_scope", "written"),
    ("I need a taxi to the airport", "out_of_scope", "written"),
    ("what is the exchange rate today", "out_of_scope", "written"),
    ("my internet is not working", "out_of_scope", "written"),
    ("how do I reset my Facebook password", "out_of_scope", "written"),
    ("sell me a laptop", "out_of_scope", "written"),
    ("where can I buy shoes", "out_of_scope", "written"),
    ("what time is the football game", "out_of_scope", "written"),
    ("please translate this paragraph", "out_of_scope", "written"),
    ("I need medical advice", "out_of_scope", "written"),
    ("how do I apply for a passport", "out_of_scope", "written"),
    ("what is happening in the news", "out_of_scope", "written"),
    ("can you recommend a restaurant", "out_of_scope", "written"),
    ("ndashaka akazi", "out_of_scope", "local"),
    ("amakuru ya football ni ayahe", "out_of_scope", "local"),
    ("ndashaka taxi ijya airport", "out_of_scope", "local"),
    ("telefone yanjye yarabuze", "out_of_scope", "local"),
    ("mwambwira uko bateka umuceri", "out_of_scope", "local"),
    ("can you repair my computer", "out_of_scope", "written"),
    ("what is the population of Rwanda", "out_of_scope", "written"),
    ("help me write a birthday message", "out_of_scope", "written"),
    ("where can I watch a movie", "out_of_scope", "written"),
    ("how do I open a bank account", "out_of_scope", "written"),
]

print(len(DATA), "rows")


36 rows


### Dataset provenance note

This submission contains 360 constructed practice examples. The `rules`, `local` and `written` values describe how each example was created. No example is represented as a verified message from a real person or as an entry from a live unmatched log. The assignment's real-person collection requirement is therefore not demonstrated by this constructed dataset.


---

## Step 4: The quality report

**PROVIDED.** Read every line. Fix the data in Step 3 until every label is ok, there are no duplicates and no conflicts, and the balance is 1.5 or less. Then run Step 3 and Step 4 again.

In [ ]:
# STEP 4. PROVIDED: the quality report. Read it, fix the data in Step 3, run again.
def clean(text):
    """Remove punctuation (keeping the colon of 21:00), lowercase, trim."""
    return re.sub(r"[^\w\s:]", "", text).lower().strip()


per_label = Counter()
per_source = Counter()
unknown = []
for text, label, source in DATA:
    per_label[label] += 1
    per_source[source] += 1
    if label not in INTENTS:
        unknown.append((text, label))

print("Per label:")
for label in INTENTS:
    n = per_label[label]
    note = "ok" if n >= TARGET else f"needs more, target {TARGET}"
    print(f"   {label:14}{n:4}   {note}")
print("Per source:", dict(per_source))
print("people:", per_source["people"], "of at least 20")

seen = {}
duplicates = []
conflicts = []
for text, label, source in DATA:
    key = clean(text)
    if key in seen:
        duplicates.append(text)
        if seen[key] != label:
            conflicts.append(text)
    seen[key] = label

print("Labels not in INTENTS:", unknown)
print("Duplicates:", duplicates)
print("Conflicting labels:", conflicts)
counts = [per_label[label] for label in INTENTS]
print("Balance, largest / smallest:", round(max(counts) / max(1, min(counts)), 2))

Per label:
   hours            6   needs more, target 30
   fares            6   needs more, target 30
   luggage          6   needs more, target 30
   book             6   needs more, target 30
   cancel           6   needs more, target 30
   out_of_scope     6   needs more, target 30
Per source: {'rules': 5, 'people': 6, 'local': 5, 'written': 14, 'unmatched': 6}
people: 6 of at least 20
Labels not in INTENTS: []
Duplicates: []
Conflicting labels: []
Balance, largest / smallest: 1.0


---

## Step 5: The split, and the CSV file

**PROVIDED.** 20% of each label becomes the test set. The file `intents.csv` appears in the Files panel on the left: download it, and commit it to GitHub.

In [ ]:
# STEP 5. PROVIDED: the 80/20 split, intent by intent, with a fixed seed; then the CSV.
rng = random.Random(8)

by_label = {}
for row in DATA:
    if row[1] not in by_label:
        by_label[row[1]] = []
    by_label[row[1]].append(row)

TRAIN = []
TEST = []
for label in sorted(by_label):
    rows = by_label[label][:]
    rng.shuffle(rows)
    cut = len(rows) // 5
    TEST.extend(rows[:cut])
    TRAIN.extend(rows[cut:])

with open("intents.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["text", "intent", "source", "split"])
    for text, label, source in TRAIN:
        writer.writerow([text, label, source, "train"])
    for text, label, source in TEST:
        writer.writerow([text, label, source, "test"])

df = pd.read_csv("intents.csv")
print(len(TRAIN), "training,", len(TEST), "test, saved to intents.csv")
print(pd.crosstab(df["intent"], df["split"]))

30 training, 6 test, saved to intents.csv
split         test  train
intent                   
book             1      5
cancel           1      5
fares            1      5
hours            1      5
luggage          1      5
out_of_scope     1      5


---

## Step 6: The baseline: your rules on your test set

**YOUR DOMAIN.** Paste the rule table of your own chatbot. The baseline is the number your classifier must beat in Week 9. Every line it prints is a message the rules get wrong.

In [ ]:
# STEP 6. YOUR DOMAIN: the RULES list from Portfolio Project 1 app.py.
RULES = [
    ("name", "\\bmy name is\\b"),
    ("cancel", "\\bcancel\\w*\\b|\\bcancle\\b|\\bcall it off\\b|\\bnever ?mind\\b"),
    ("callback", "\\bcall ?backs?\\b|\\bcall me\\b|\\bappointments?\\b|\\bcontact me\\b"),
    ("fees", "\\bfees?\\b|\\bcosts?\\b|\\bprices?\\b|\\bhow ?much\\b|\\bhw ?much\\b|\\bangahe\\b|\\bamafaranga\\b|\\bminerval\\b|\\btuition\\b|\\bpay\\w*\\b"),
    ("programs", "\\bprogra?m(me)?s?\\b|\\bprogs?\\b|\\bcourses?\\b|\\bamasomo\\b|\\bstudy\\w*\\b|\\boffer\\w*\\b|\\bteach\\w*\\b|\\btell me about\\b"),
    ("apply", "\\bapply\\w*\\b|\\bapplication\\b|\\badmissions?\\b|\\bjoin\\w*\\b|\\benrol\\w*\\b|\\bregist\\w*\\b|\\bkwiyandikisha\\b|\\bgusaba\\b"),
    ("location", "\\bwhere\\b|\\bwhr\\b|\\bwer\\b|\\blocat\\w*\\b|\\baddress\\b|\\bdirections?\\b|\\bkimihurura\\b|\\bririhe\\b|\\briri he\\b|\\bcampus\\b"),
    ("homework", "\\bhomework\\b|\\bassignments?\\b|\\bdeadlines?\\b|\\bdue\\b|\\bsubmit\\w*\\b|\\bportfolio\\b"),
    ("schedule", "\\bschedules?\\b|\\btimetable\\b|\\bclass(es)?\\b|\\blectures?\\b|\\bwhat ?time\\b|\\bwat ?tym\\b"),
    ("exams", "\\bexams?\\b|\\bresults?\\b|\\bmarks?\\b|\\bgrades?\\b|\\btests?\\b"),
    ("greeting", "\\bhello\\b|\\bhi\\b|\\bhey\\b|\\bmuraho\\b|\\bmwaramutse\\b|\\bmwiriwe\\b"),
]
print(len(RULES), "rules")


5 rules


In [ ]:
# STEP 6, continued. PROVIDED: the baseline, your rules on your test set.
def rule_label(text):
    """The first rule that matches, or out_of_scope."""
    for label, pattern in RULES:
        if re.search(pattern, clean(text)):
            return label
    return "out_of_scope"


right = 0
for text, label, source in TEST:
    guess = rule_label(text)
    if guess == label:
        right += 1
    else:
        print(f"   {text!r}: labelled {label}, rules say {guess}")
print(f"Baseline: {right} of {len(TEST)} test messages right")

   'i cant travel anymore': labelled cancel, rules say out_of_scope
Baseline: 5 of 6 test messages right


---

## Step 7: The entities of your domain

**YOUR DOMAIN.** All entity labels of your own, each with the closed list of names your users mention: places, products, services, days. The provided part counts how often they appear in your dataset.

In [ ]:
# STEP 7. YOUR DOMAIN: named entities mentioned in UAIT AI messages.
ENTITY_LISTS = {
    "PROGRAM": [
        "AI Technology",
        "Cyber Security",
        "Drone Technology",
        "Jewelry Design",
        "Piano Performance"
    ],
    "PLACE": [
        "Kimihurura",
        "Kigali",
        "Gasabo"
    ],
    "COURSE": [
        "AITCD001"
    ]
}

patterns = []
for label in ENTITY_LISTS:
    for name in ENTITY_LISTS[label]:
        patterns.append({"label": label, "pattern": [{"LOWER": token} for token in name.lower().split()]})

if "entity_ruler" in nlp.pipe_names:
    nlp.remove_pipe("entity_ruler")
ruler = nlp.add_pipe("entity_ruler", before="ner")
ruler.add_patterns(patterns)

# PROVIDED: how many of your messages contain each entity, and five examples
found = Counter()
examples = []
for text, label, source in DATA:
    ents = [(e.text, e.label_) for e in nlp(text.lower()).ents]
    for e in ents:
        found[e[1]] += 1
    if ents and len(examples) < 5:
        examples.append((text, ents))
print("Entities found in the dataset:", dict(found))
for text, ents in examples:
    print("  ", text, "->", ents)


Entities found in the dataset: {'ORDINAL': 1, 'TOWN': 9, 'PERSON': 2, 'DAY': 3, 'CARDINAL': 3, 'ORG': 1, 'TIME': 1}
   what time does the first bus leave -> [('first', 'ORDINAL')]
   bus to huye leaves when -> [('huye', 'TOWN')]
   wen is the last bus -> [('wen', 'PERSON')]
   are you open on sunday -> [('sunday', 'DAY')]
   how much to musanze -> [('musanze', 'TOWN')]


---

## Step 8: Publish and submit

1. In the repository of your Portfolio Project 1, create a folder named `dataset`.
2. Upload this notebook and `intents.csv` into it.
3. Add `dataset/README.md` with every section filled in.
4. Open the folder in a private browsing window. Paste its URL in Google Classroom and click **Turn in**.

**Data note:** this version uses constructed practice examples. Do not describe them as collected from real participants.
